# 数组读写与内存映射

学习目标：保存和恢复数组，按文本格式处理表头、类型与缺失值，理解内存映射并正确关闭文件资源。

前置知识：文件与路径、with 语句、dtype、数组形状、切片、视图与副本。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用自行生成的小文件，每个实验在 TemporaryDirectory 中读写，退出 with 后自动清理。后续单元沿用首次导入的 np、Path 和 TemporaryDirectory。

## 1 保存与恢复一个数组

把两台设备的三次整数读数保存下来，再恢复为数组，可以使用 save 和 load。save 写入 .npy 格式，文件包含恢复普通数组所需的数值、shape 和 dtype。

Path 表示文件路径，TemporaryDirectory 创建本次实验的临时目录。下面的 readings.npy 只在这个目录中创建，不覆盖已有数据；退出 with 后文件和目录一起删除。

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np

readings = np.array([[18, 19, 20], [21, 22, 23]], dtype=np.int16)

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "readings.npy"
    np.save(file_path, readings, allow_pickle=False)
    restored = np.load(file_path, allow_pickle=False)
    print(restored)  # 两行各三次读数，数值与 readings 相同。
    print(restored.shape, restored.dtype)  # (2, 3)，int16。
    print(np.array_equal(restored, readings))  # True：形状与数值一致。
    print(restored.dtype == readings.dtype)  # True：另行检查元素类型。

print(file_path.exists())  # False：退出临时目录后文件已删除。

[[18 19 20]
 [21 22 23]]
(2, 3) int16
True
True
False


## 2 在一个文件中保存多个数组

读数和设备编号的形状、类型不同时，可以用 savez 放进同一个 .npz 文件，分别保留各数组的信息。关键字参数作为数组名称；如果只传位置参数，名称会成为 arr_0、arr_1 等。

load 读取 .npz 时返回 NpzFile，它按名称读取其中的数组，需要关闭。使用 with 可以在离开代码块时关闭归档；已经读出的普通数组仍可继续使用。

| 方法 | 中文名称／含义 | 保存内容 |
| --- | --- | --- |
| save | 单数组保存 | 一个 .npy 数组 |
| savez | 多数组归档 | 一个未压缩的 .npz 归档 |
| savez_compressed | 压缩多数组归档 | 一个压缩的 .npz 归档 |
| load | 数组或归档读取 | 读取 .npy，或打开 .npz |

In [2]:
device_ids = np.array([101, 102], dtype=np.int32)
readings = np.array([[18.5, 19.0, 20.5], [21.0, 22.5, 23.0]], dtype=np.float32)

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "inspection.npz"
    np.savez(file_path, ids=device_ids, values=readings, allow_pickle=False)
    with np.load(file_path, allow_pickle=False) as archive:
        print(archive.files)  # ['ids', 'values']：两个具名数组。
        restored_ids = archive["ids"]
        restored_values = archive["values"]

    print(restored_ids)  # [101 102]，归档关闭后仍可使用已读出的数组。
    print(restored_ids.shape, restored_ids.dtype)  # (2,)，int32。
    print(restored_values.shape, restored_values.dtype)  # (2, 3)，float32。
    print(np.array_equal(restored_values, readings))  # True

print(file_path.exists())  # False：归档关闭后完成清理。

['ids', 'values']
[101 102]
(2,) int32
(2, 3) float32
True
False


savez_compressed 使用压缩归档，读取方法相同。是否值得压缩取决于数据和读写需求；下面只核对恢复结果，不根据几个元素推断大文件的压缩率或性能。

In [3]:
values = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.int16)

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "compressed.npz"
    np.savez_compressed(file_path, values=values, allow_pickle=False)
    with np.load(file_path, allow_pickle=False) as archive:
        restored = archive["values"]
    print(restored)  # 恢复原来的两行三列整数。
    print(restored.shape, restored.dtype)  # (2, 3)，int16。
    print(np.array_equal(restored, values))  # True

[[1 2 3]
 [4 5 6]]
(2, 3) int16
True


## 3 对象数组与 allow_pickle

普通数值数组不需要 pickle。含 Python 对象的数组需要 pickle 才能写入这类文件；读取 pickle 可能执行任意代码，因此不能为读取不可信文件而随意开启 allow_pickle。

load 默认 allow_pickle=False，本章仍显式写出这个条件。下面只保存自己创建的对象数组用于演示拒绝读取，不开启反序列化。文件扩展名为 .npy 本身不能证明其中只有普通数值。

In [4]:
objects = np.array([{"device": 101}], dtype=object)

# 预期 ValueError：禁用 pickle 时不能读取这个对象数组；退出时清理临时文件。
with TemporaryDirectory() as directory:
    file_path = Path(directory) / "objects.npy"
    np.save(file_path, objects, allow_pickle=True)  # 仅创建本例自己控制的文件。
    np.load(file_path, allow_pickle=False)

ValueError: Object arrays cannot be loaded when allow_pickle=False

## 4 导出和读取文本

### 4.1 分隔符、格式与编码

需要直接查看数值或与其他工具交换简单表格时，可以用 savetxt 写文本，用 loadtxt 读回。savetxt 接收一维或二维数组；下面每行是一台设备，每列分别为温度和电压。

delimiter="," 用逗号分列，fmt="%.2f" 写出小数点后两位，header 写表头。默认 comments="# " 给表头加注释前缀，loadtxt 会跳过它。写入和读取都指定 UTF-8，以正确处理中文表头。

In [5]:
readings = np.array([[20.25, 3.50], [21.75, 3.25], [19.50, 3.00]])

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "readings.csv"
    np.savetxt(
        file_path, readings, delimiter=",", fmt="%.2f",
        header="温度,电压", encoding="utf-8",
    )
    print(file_path.read_text(encoding="utf-8"))  # 中文注释表头，加三行两列数值。
    restored = np.loadtxt(file_path, delimiter=",", encoding="utf-8", ndmin=2)
    print(restored.shape, restored.dtype)  # (3, 2)，默认 float64。
    print(np.array_equal(restored, readings))  # True：本例数值可由两位小数完整表示。

# 温度,电压
20.25,3.50
21.75,3.25
19.50,3.00

(3, 2) float64
True


### 4.2 普通表头与指定列

如果表头没有注释前缀，loadtxt 可以用 skiprows=1 跳过第一行。usecols 按从 0 开始的列号选择列。下面读取温度和电压，跳过设备编号列。

loadtxt 适合列数规则、能够按指定类型转换的文本；分隔符只支持单个字符。它不会根据文件名中的 .csv 自动决定分隔符。

In [6]:
text = "设备,温度,电压\n101,20.25,3.50\n102,21.75,3.25\n"

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "selected-columns.csv"
    file_path.write_text(text, encoding="utf-8")
    readings = np.loadtxt(
        file_path, delimiter=",", skiprows=1, usecols=(1, 2),
        dtype=np.float64, encoding="utf-8", ndmin=2,
    )
    print(readings)  # [[20.25 3.50], [21.75 3.25]]，列顺序与 usecols 一致。
    print(readings.shape, readings.dtype)  # (2, 2)，float64。

[[20.25  3.5 ]
 [21.75  3.25]]
(2, 2) float64


## 5 文本往返的类型与精度

### 5.1 原来的类型和形状不会自动恢复

文本只写出了格式化数值，不能据此自动还原原数组的 dtype 和全部形状信息。loadtxt 默认读取为浮点数，并压缩单例轴。需要保持二维表格时使用 ndmin=2；需要整数类型时，按文件约定显式指定 dtype。

In [7]:
counts = np.array([[10, 20, 30]], dtype=np.int16)

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "counts.csv"
    np.savetxt(file_path, counts, delimiter=",", fmt="%d", encoding="utf-8")
    default_result = np.loadtxt(file_path, delimiter=",", encoding="utf-8")
    typed_result = np.loadtxt(
        file_path, delimiter=",", dtype=np.int16, ndmin=2, encoding="utf-8",
    )
    print(default_result)  # [10. 20. 30.]：数值相同，但表示不同。
    print(default_result.shape, default_result.dtype)  # (3,)，float64。
    print(typed_result.shape, typed_result.dtype)  # (1, 3)，int16。
    print(np.array_equal(typed_result, counts))  # True

[10. 20. 30.]
(3,) float64
(1, 3) int16
True


### 5.2 写出的位数限制了恢复精度

fmt 决定写入文本的数字。若只写两位小数，读回时即使使用 float64，也无法恢复被舍去的小数位。下面用已知输入观察误差；需要保留原始数组时优先使用 .npy。

In [8]:
readings = np.array([1.23456, 2.34567], dtype=np.float64)

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "rounded.csv"
    np.savetxt(file_path, readings, fmt="%.2f", encoding="utf-8")
    restored = np.loadtxt(file_path, encoding="utf-8")
    print(restored)  # [1.23 2.35]：只保留写出的两位小数。
    print(np.abs(restored - readings))  # 误差约为 0.00456、0.00433。
    print(np.array_equal(restored, readings))  # False：提高读取类型精度也无法补回信息。

[1.23 2.35]
[0.00456 0.00433]
False


## 6 缺失值与不规则输入

### 6.1 指定缺失标记和填充值

表格有空字段或特定缺失标记时，使用 genfromtxt。missing_values 指定额外的缺失字符串，空字段也会识别为缺失；filling_values 指定填入的数值。

下面把空字段和 NA 都读为 NaN，dtype 使用浮点类型。skip_header 是 genfromtxt 跳过表头的参数，名称与 loadtxt 的 skiprows 不同。loose=False 要求无法转换且未声明为缺失的内容报错。

In [9]:
text = "温度,电压\n20.5,3.3\n,3.2\n21.0,NA\n"

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "missing.csv"
    file_path.write_text(text, encoding="utf-8")
    readings = np.genfromtxt(
        file_path, delimiter=",", skip_header=1, dtype=np.float64,
        missing_values="NA", filling_values=np.nan, loose=False,
        encoding="utf-8", ndmin=2,
    )
    print(readings)  # 第二行温度、第三行电压为 nan，其他值保留。
    print(readings.shape, readings.dtype)  # (3, 2)，float64。
    print(np.isnan(readings))  # 两个缺失位置为 True。

[[20.5  3.3]
 [ nan  3.2]
 [21.   nan]]
(3, 2) float64
[[False False]
 [ True False]
 [False  True]]


### 6.2 整数中的缺失约定

整数数组不能用 NaN 表示缺失。若计数的有效范围是非负整数，可以约定用 -1 表示缺失，并保留这个约定。填入 -1 不会自动产生掩码，后续计算仍须识别并排除它。

In [10]:
text = "3,4\n,5\n"

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "missing-counts.csv"
    file_path.write_text(text, encoding="utf-8")
    counts = np.genfromtxt(
        file_path, delimiter=",", dtype=np.int16, filling_values=-1,
        loose=False, encoding="utf-8", ndmin=2,
    )
    print(counts)  # [[3 4], [-1 5]]：-1 为本例约定的缺失标记。
    print(counts == -1)  # 只有第二行第一列为 True。
    print(counts.shape, counts.dtype)  # (2, 2)，int16。

[[ 3  4]
 [-1  5]]
[[False False]
 [ True False]]
(2, 2) int16


### 6.3 缺失字段不等于列数错误

逗号之间的空字段有明确位置；整行少一列则不能知道哪一列缺失。genfromtxt 默认 invalid_raise=True，对列数不一致报错。不要为了让程序继续运行而自动跳过这些行。

下面分别观察少一列，以及把普通文字误放进数值列。两者都应先修正输入或明确规则。

In [11]:
# 预期 ValueError：第一行两列，第二行只有一列。
with TemporaryDirectory() as directory:
    file_path = Path(directory) / "invalid.csv"
    file_path.write_text("1,2\n3\n", encoding="utf-8")
    np.genfromtxt(file_path, delimiter=",", encoding="utf-8")

ValueError: Some errors were detected !
    Line #2 (got 1 columns instead of 2)

In [12]:
# 预期 ValueError：wrong 未声明为缺失，严格转换不能把它转成数值。
with TemporaryDirectory() as directory:
    file_path = Path(directory) / "invalid.csv"
    file_path.write_text("1,2\n3,wrong\n", encoding="utf-8")
    np.genfromtxt(file_path, delimiter=",", loose=False, encoding="utf-8")

ValueError: Cannot convert string 'wrong'

## 7 选学：原始二进制与元数据
已有原始二进制文件时，可用 fromfile 按约定的 dtype 解释字节。tofile 与 fromfile 的原始二进制往返不保存 dtype、字节序和 shape，需要双方另行约定；不能把它当作自带这些信息的 .npy。

下面的 >i2 表示大端、两个字节的有符号整数，<i2 表示小端。大端把高位字节放在前面，小端把低位字节放在前面。用错误字节序读取，即使文件和元素数都没变，也会得到错误数值。

In [13]:
values = np.array([[1, 256, 513], [2, 512, 1025]], dtype=">i2")

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "raw.bin"
    values.tofile(file_path)
    restored = np.fromfile(file_path, dtype=">i2").reshape((2, 3))
    wrong_order = np.fromfile(file_path, dtype="<i2")
    print(restored)  # 恢复原来的两行三列整数。
    print(restored.shape, restored.dtype)  # (2, 3)，>i2。
    print(wrong_order[:3])  # [256 1 258]：前三个数已被错误解释。
    print(np.array_equal(restored, values))  # True

[[   1  256  513]
 [   2  512 1025]]
(2, 3) >i2
[256   1 258]
True


## 8 选学：内存映射
内存映射可以访问大文件的一小段，而不先把整个文件读成普通内存数组。原始二进制可用 np.memmap(file_path, dtype=..., shape=..., mode="r")，其中 file_path 是文件路径，dtype 和 shape 按文件约定填写；.npy 可用 np.load(file_path, mmap_mode="r", allow_pickle=False)，由文件头提供形状和类型。

| 模式 | 中文名称／含义 | 文件行为 |
| --- | --- | --- |
| r | 只读 | 读取已有文件，不允许修改 |
| r+ | 读写 | 修改已有文件 |
| w+ | 新建或覆盖 | 创建文件或覆盖原内容，需明确形状 |
| c | 写时复制 | 修改内存中的数据，不写回原文件 |

memmap.flush 将修改写回文件，但不是关闭操作。NumPy 的 memmap 没有公开的底层映射关闭接口，且视图可能共享映射。需要明确控制关闭时，可以按官方建议，用 Python 的 mmap 对象作为 ndarray 的 buffer。

下面用小文件观察这个机制：ACCESS_WRITE 允许写回文件，映射长度 0 表示整个已有文件。保留的片段先 copy，再在 finally 中释放依赖映射的数组，随后退出两层 with，依次关闭映射和文件。映射关闭后不再访问原数组或其视图。

In [14]:
import mmap

readings = np.array([[10.0, 11.0, 12.0], [20.0, 21.0, 22.0]], dtype=np.float64)

# 1. 先写出二进制文件；映射数组的形状和 dtype 必须与写入布局对应。
with TemporaryDirectory() as directory:
    file_path = Path(directory) / "mapped.bin"
    readings.tofile(file_path)
    # 2. 文件、映射、数组依次建立；退出时按相反顺序释放。
    with file_path.open("r+b") as file:
        with mmap.mmap(file.fileno(), 0, access=mmap.ACCESS_WRITE) as mapping:
            mapped = np.ndarray((2, 3), dtype=np.float64, buffer=mapping)
            try:
                # 需要带出映射范围的数据先复制，不能保留依赖映射的视图。
                fragment = mapped[1, :2].copy()
                print(fragment)  # [20. 21.]，副本可在映射关闭后使用。
                print(mapped.shape, mapped.dtype)  # (2, 3)，float64。
                mapped[0, 0] = 99.0
                # 将修改写回文件；flush 本身不负责关闭映射。
                mapping.flush()
            finally:
                del mapped  # 释放依赖映射的数组，不保留它的切片视图。

    print(mapping.closed, file.closed)  # True True：映射和文件都已关闭。
    # 3. 映射已经关闭，重新读取文件观察写回结果。
    restored = np.fromfile(file_path, dtype=np.float64).reshape((2, 3))
    print(restored)  # 第一行第一列变为 99，其余读数保留。
    print(fragment)  # [20. 21.]：保留的是独立副本。

print(file_path.exists())  # False：Windows 上也已完成临时文件清理。

[20. 21.]
(2, 3) float64
True True
[[99. 11. 12.]
 [20. 21. 22.]]
[20. 21.]
False


## 本章小结

（1）.npy 保存单个普通数组的数值、形状和类型；.npz 按名称保存多个数组，读取后要关闭归档。

（2）文本读写要约定分隔符、表头、编码、dtype 和形状；写出位数不足造成的信息损失，不能靠提高读取精度补回。

（3）genfromtxt 可以识别缺失标记并填值；缺失字段、列数错误和无法转换的内容需要分别处理。

（4）不可信对象数组不能通过开启 allow_pickle 来强行读取；原始二进制的 dtype、字节序和形状也不能靠猜测。

（5）内存映射适合局部访问，但要管理映射与视图的生命周期；刷新修改和关闭资源是不同操作。

## 练习

（1）将下面数组保存为 .npy，再读回。分别检查数值、shape 和 dtype，确认退出临时目录后文件不存在。不能只凭数值相同就判断完整往返成功。

In [15]:
values = np.array([[2, 4, 6], [8, 10, 12]], dtype=np.int32)

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "exercise.npy"
    # 在此保存和读取；显式关闭 pickle，打印三项检查结果。

# 在此检查临时文件已经清理。

（2）先预测同一个文本文件默认读取与指定参数读取后的 shape 和 dtype，再运行核对。解释为什么保存前的一行二维整数数组可能读成一维浮点数组。

In [16]:
values = np.array([[7, 8, 9]], dtype=np.int16)

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "exercise.csv"
    np.savetxt(file_path, values, delimiter=",", fmt="%d", encoding="utf-8")
    first = np.loadtxt(file_path, delimiter=",", encoding="utf-8")
    second = np.loadtxt(
        file_path, delimiter=",", dtype=np.int16, ndmin=2, encoding="utf-8",
    )
    # 先记录预测，再运行下面两行核对。
    print(first.shape, first.dtype)
    print(second.shape, second.dtype)

(3,)

 float64
(1, 3) int16


（3）任务最初要求给人查看，只保留两位小数；后来改为保存两个不同形状的数组，要求恢复原始 dtype 和数值。分别选择格式与读写方法，用注释解释理由，并完成第二个任务。说明为什么原来的文本输出不再满足新条件。

In [17]:
device_ids = np.array([101, 102], dtype=np.int32)
readings = np.array([[1.23456, 2.34567], [3.45678, 4.56789]], dtype=np.float64)

# 在此说明两种需求对应的格式，以及文本精度损失。
with TemporaryDirectory() as directory:
    file_path = Path(directory) / "exercise.npz"
    # 在此保存具名数组，用 with 读取归档，检查各自的 shape、dtype 和数值。

（4）读取带空字段和 NA 的温度记录，保留两列和三行，用 NaN 表示缺失。检查缺失位置，不把它们自动改为 0。若任务改成只接受完整记录，说明应怎样处理，而不是只修改读取函数让错误消失。

In [18]:
text = "早间,晚间\n18.5,20.0\nNA,21.0\n19.0,\n"

with TemporaryDirectory() as directory:
    file_path = Path(directory) / "exercise-missing.csv"
    file_path.write_text(text, encoding="utf-8")
    # 在此读取并打印结果、shape、dtype 和缺失掩码。
    # 检查：两个缺失位置；用注释说明只接受完整记录时的处理选择。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（2.5） | [Reading and writing files](https://numpy.org/doc/2.5/user/how-to-io.html)：文本、NumPy 格式与大数组读写的选择；[save](https://numpy.org/doc/2.5/reference/generated/numpy.save.html)、[load](https://numpy.org/doc/2.5/reference/generated/numpy.load.html)：文件参数、allow_pickle、mmap_mode、返回值与关闭归档；[NPY format](https://numpy.org/doc/2.5/reference/generated/numpy.lib.format.html)：Capabilities、shape 与 dtype；[savez](https://numpy.org/doc/2.5/reference/generated/numpy.savez.html)、[savez_compressed](https://numpy.org/doc/2.5/reference/generated/numpy.savez_compressed.html)、[NpzFile](https://numpy.org/doc/2.5/reference/generated/numpy.lib.npyio.NpzFile.html)：数组命名、压缩与延迟读取；[savetxt](https://numpy.org/doc/2.5/reference/generated/numpy.savetxt.html)：fmt、header、comments、encoding；[loadtxt](https://numpy.org/doc/2.5/reference/generated/numpy.loadtxt.html)：dtype、delimiter、skiprows、usecols、ndmin；[genfromtxt](https://numpy.org/doc/2.5/reference/generated/numpy.genfromtxt.html) 与 [Importing data](https://numpy.org/doc/2.5/user/basics.io.genfromtxt.html)：missing_values、filling_values、loose、invalid_raise、缺失值处理；[tofile](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.tofile.html)、[fromfile](https://numpy.org/doc/2.5/reference/generated/numpy.fromfile.html)：原始二进制与缺失的元数据；[dtype](https://numpy.org/doc/2.5/reference/arrays.dtypes.html)：字节序和类型字符串；[memmap](https://numpy.org/doc/2.5/reference/generated/numpy.memmap.html)：模式、flush、关闭限制与 mmap 替代方式；[ndarray](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.html)：buffer 构造参数。 |
| Python 官方文档（3.12） | [tempfile](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)：临时目录上下文与清理；[pathlib](https://docs.python.org/3.12/library/pathlib.html#reading-and-writing-files)：open、read_text、write_text；[mmap](https://docs.python.org/3.12/library/mmap.html)：ACCESS_WRITE、上下文管理器、flush、close 和 closed。 |